# CyberSentinel-EU - Baseline Pipeline (v0)
**Fatema Husain Hasan (202508958) - MSc AI Thesis, Bahrain Polytechnic**

**Purpose:** working baseline for the Mid-Review - keyword screening of the GDPR
Enforcement Tracker snapshot, manually reviewed starter labels (80 records),
and a binary classifier (cyber vs non-cyber) trained on **structured features only**.

**Note on tooling:** PyCaret is incompatible with Colab's Python 3.12 runtime
(PyCaret requires ≤3.11), so model comparison is implemented with scikit-learn.
Migration decision (local PyCaret env vs scikit-learn throughout) to be agreed
with the supervisor - see risk register.

Run top to bottom. Runtime: CPU.

## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
print('pandas', pd.__version__, '| numpy', np.__version__)

pandas 2.2.2 | numpy 2.0.2


## 2. Load data from GitHub
Both files load directly from the public repository - no manual uploads needed.

In [2]:
REPO = "https://raw.githubusercontent.com/Fatimaxx24/202508958_IT9099_Thesis/main/"

# 2a. Full Tracker snapshot (3,202 records, extracted 15 Jul 2026)
df = pd.read_csv(REPO + "gdpr_enforcement_tracker_full.csv", low_memory=False)
print('Tracker snapshot:', df.shape)

# 2b. Manually reviewed starter labels (80 records, labelled 16 Jul 2026)
labels = pd.read_csv(REPO + "labelling_sample_80_REVIEWED.csv")
print('Labelled sample :', labels.shape)

Tracker snapshot: (3202, 12)
Labelled sample : (80, 11)


## 3. Task 1 - Keyword screening of Summary text
Three keyword groups flag candidate cyber records. This screening is used for
**sampling and audit prioritisation only** - final labels come from manual review.

In [3]:
MALICIOUS = ['ransomware','phishing','hacker','hacking','hacked','cyberattack',
             'cyber attack','cyber-attack','malware','brute force','credential',
             'exfiltrat','ddos','denial of service','sql injection','compromis',
             'intrusion','breach by third part','stolen credentials','dark web']
NONMAL_CYBER = ['accidental disclosure','sent to the wrong','wrong recipient',
                'misconfigur','unprotected database','publicly accessible',
                'exposed database','unsecured','technical error','software error']
AMBIG = ['unauthorized access','unauthorised access','data breach','security incident',
         'personal data breach','leak']

s = df['Summary'].astype(str).str.lower()
def flag(series, kws):
    return series.str.contains('|'.join(re.escape(k) for k in kws), regex=True)

df['kw_malicious'] = flag(s, MALICIOUS)
df['kw_nonmal']    = flag(s, NONMAL_CYBER)
df['kw_ambig']     = flag(s, AMBIG)
df['kw_any_cyber'] = df.kw_malicious | df.kw_nonmal | df.kw_ambig

print(f"records with any cyber keyword: {df.kw_any_cyber.sum()} "
      f"({df.kw_any_cyber.mean()*100:.1f}% of corpus)")

records with any cyber keyword: 482 (15.1% of corpus)


## 4. Task 2 - Prepare the labelled training set
Manual labels: `cyber_malicious` / `cyber_nonmalicious` / `not_cyber` / `unclear`.
Records labelled `unclear` are excluded. Binary target: **is_cyber**.

In [4]:
valid = ['cyber_malicious','cyber_nonmalicious','not_cyber']
lab = labels[labels.manual_label.isin(valid)].copy()
lab['is_cyber'] = lab.manual_label.str.startswith('cyber').astype(int)

print('usable labelled records:', len(lab),
      '| cyber share:', lab.is_cyber.mean().round(2))
print(lab.manual_label.value_counts().to_string())

usable labelled records: 74 | cyber share: 0.8
manual_label
cyber_malicious       31
cyber_nonmalicious    28
not_cyber             15


## 5. Task 3 - Structured feature engineering
v0 uses structured features only (country, sector, year, log fine, article count).
Text embeddings are Sprint 2 - keeping text out of v0 avoids any circularity with
the keyword-based sampling.

In [5]:
d = df.merge(lab[['ETid','is_cyber']], on='ETid', how='inner')
d['year']       = pd.to_datetime(d['Date of Decision'], errors='coerce',
                                 format='mixed').dt.year
d['log_fine']   = np.log1p(d['Fine (EUR)'])
d['n_articles'] = d['Quoted Articles'].astype(str).str.count('Art')

data = d[['Country','Sector','year','log_fine','n_articles','is_cyber']].copy()
data['year']     = data['year'].fillna(data['year'].median())
data['log_fine'] = data['log_fine'].fillna(-1)
print('training table:', data.shape)
data.head()

training table: (74, 6)


,Country,Sector,year,log_fine,n_articles,is_cyber
0,Germany,"Media, Telecoms and Broadcasting",2018,9.903538,1,1
1,Romania,Industry and Commerce,2019,8.006701,1,1
2,France,"Finance, Insurance and Consulting",2019,12.100718,1,1
3,Bulgaria,Public Sector and Education,2019,14.771022,1,1
4,Norway,Public Sector and Education,2019,11.695255,1,1


## 6. Task 4 - Model comparison (5-fold stratified CV)
Six classifier families compared on accuracy, macro-F1 (primary metric), and AUC.

In [7]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              ExtraTreesClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

X = data.drop(columns='is_cyber')
y = data['is_cyber']

prep = ColumnTransformer(
    [('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
      ['Country','Sector'])],
    remainder='passthrough')

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting'  : GradientBoostingClassifier(random_state=42),
    'Extra Trees'        : ExtraTreesClassifier(n_estimators=200, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(random_state=42),
    'Naive Bayes'        : GaussianNB(),
}

rows, cv = [], StratifiedKFold(5, shuffle=True, random_state=42)
for name, m in models.items():
    pipe = Pipeline([('prep', prep), ('model', m)])
    sc = cross_validate(pipe, X, y, cv=cv,
                        scoring=['accuracy','f1_macro','roc_auc'])
    rows.append({'Model': name,
                 'Accuracy': sc['test_accuracy'].mean().round(3),
                 'Macro F1': sc['test_f1_macro'].mean().round(3),
                 'AUC'     : sc['test_roc_auc'].mean().round(3)})

results = pd.DataFrame(rows).sort_values('Macro F1',
                        ascending=False).reset_index(drop=True)
results

,Model,Accuracy,Macro F1,AUC
0,Decision Tree,0.892,0.816,0.808
1,Gradient Boosting,0.866,0.755,0.816
2,Extra Trees,0.825,0.684,0.836
3,Random Forest,0.826,0.668,0.820
4,Naive Bayes,0.676,0.620,0.742
5,Logistic Regression,0.783,0.471,0.785


## 7. Notes for the Mid-Review
- **Baseline v0**: binary cyber vs non-cyber, structured features only,
  74 manually reviewed labels (6 'unclear' excluded).
- The 80-record sample was **stratified by keyword signal**, so its class balance
  does not estimate the corpus-wide cyber rate (keyword screening: 15.1%).
- Manual review found keyword **false positives** (media/telemarketing cases) and
  at least one **false negative** (SIM-swap fraud) - motivating the multi-signal
  labelling protocol.
- Next sprints: full 200-record audit → ICO integration → Azure OpenAI embeddings
  (hybrid features) → grouped CV by organisation → leakage ablations → RAG stage.